# Inference Playground

Plug-and-play inference on any audio folder. Automatically:
1. Scans folder for audio files
2. Loads or creates transcriptions, text features, WavLM features, wav2vec2 scores
3. Runs weighted ensemble prediction
4. If GT labels exist, computes metrics

**Set weights to 0 to skip that model entirely** (won't compute features if not needed).

In [ ]:
# ================================================================
# CONFIGURATION — edit these paths and weights
# ================================================================

# Input folder containing audio files
AUDIO_FOLDER = r"audios2"  # e.g. "audios2", "audios4", or full path

# Model paths (set to None or "" to skip)
TEXT_XGBOOST_MODEL  = r"checkpoints_finetuned/xgboost_finetuned.json"
TEXT_SCALER         = r"checkpoints_finetuned/scaler_finetuned.pkl"
WAVLM_XGBOOST_MODEL = r"checkpoints_wavlm/xgboost_wavlm.json"
WAVLM_SCALER        = r"checkpoints_wavlm/scaler_wavlm.pkl"
WAV2VEC2_ONNX       = r"checkpoints_unified/wav2vec2_trained_quant.onnx"

# Weights — must sum to 1.0 (set to 0 to disable that model)
W_TEXT   = 0.4
W_WAVLM  = 0.3
W_WAV2VEC2 = 0.3

# Threshold for final prediction
THRESHOLD = 0.45

# Whisper model for transcription (only used if transcripts don't exist)
WHISPER_MODEL = "small"  # "tiny", "base", "small", "medium"

# wav2vec2 settings
WAV2VEC2_THRESHOLD = 0.65  # P(read) threshold per 5-sec window

In [ ]:
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import joblib

warnings.filterwarnings('ignore')

# Validate weights
assert abs(W_TEXT + W_WAVLM + W_WAV2VEC2 - 1.0) < 0.01, \
    f"Weights must sum to 1.0, got {W_TEXT + W_WAVLM + W_WAV2VEC2}"

AUDIO_EXTS = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".wma", ".aac", ".webm", ".mp4"}

## 1. Scan Folder

In [ ]:
audio_dir = Path(AUDIO_FOLDER).resolve()
folder_name = audio_dir.name

# Find all audio files
audio_files = sorted([
    f for f in audio_dir.rglob("*")
    if f.suffix.lower() in AUDIO_EXTS and f.is_file()
])

print(f"Folder: {audio_dir}")
print(f"Audio files found: {len(audio_files)}")
for ext in sorted(set(f.suffix.lower() for f in audio_files)):
    n = sum(1 for f in audio_files if f.suffix.lower() == ext)
    print(f"  {ext}: {n}")

# Check for GT labels: {foldername}GT.csv
gt_candidates = [
    audio_dir / f"{folder_name}GT.csv",
    audio_dir.parent / f"{folder_name}GT.csv",
    audio_dir / "GT.csv",
    audio_dir / "labels.csv",
]
GT_CSV = None
for gp in gt_candidates:
    if gp.exists():
        GT_CSV = gp
        break

HAS_GT = GT_CSV is not None
if HAS_GT:
    gt_df = pd.read_csv(GT_CSV)
    print(f"\nGround truth: {GT_CSV}")
    print(f"  Columns: {list(gt_df.columns)}")
    print(f"  Rows: {len(gt_df)}")
    # Try to find label column
    label_col = None
    for candidate in ["label_int", "label", "gt", "GT", "ground_truth", "cheating"]:
        if candidate in gt_df.columns:
            label_col = candidate
            break
    if label_col:
        print(f"  Label column: {label_col}")
        print(f"  {gt_df[label_col].value_counts().to_dict()}")
    else:
        print(f"  WARNING: No label column found. Will treat as unlabeled.")
        HAS_GT = False
else:
    print(f"\nNo GT file found (looked for {folder_name}GT.csv). Running without labels.")

# Derived paths
TRANSCRIPTS_JSON = audio_dir.parent / f"{folder_name}_transcripts.json"
FEATURES_CSV     = audio_dir.parent / f"{folder_name}_features.csv"
WAVLM_FEAT_CSV   = audio_dir.parent / f"{folder_name}_wavlm.csv"

print(f"\nDerived paths:")
print(f"  Transcripts: {TRANSCRIPTS_JSON} {'[EXISTS]' if TRANSCRIPTS_JSON.exists() else '[MISSING]'}")
print(f"  Text features: {FEATURES_CSV} {'[EXISTS]' if FEATURES_CSV.exists() else '[MISSING]'}")
print(f"  WavLM features: {WAVLM_FEAT_CSV} {'[EXISTS]' if WAVLM_FEAT_CSV.exists() else '[MISSING]'}")

## 2. Transcription (if needed)
Skipped if transcripts JSON already exists or if text weight is 0.

In [ ]:
need_transcripts = (W_TEXT > 0) and not TRANSCRIPTS_JSON.exists()

if need_transcripts:
    import whisper
    import librosa

    print(f"Transcribing {len(audio_files)} files with Whisper ({WHISPER_MODEL})...")
    model_w = whisper.load_model(WHISPER_MODEL)

    transcripts = {}
    for fp in tqdm(audio_files, desc="Transcribing"):
        try:
            result = model_w.transcribe(
                str(fp), language="en", word_timestamps=True,
                condition_on_previous_text=True, fp16=False,
            )
            words = []
            for seg in result.get("segments", []):
                for w in seg.get("words", []):
                    words.append({
                        "word": w.get("word", "").strip(),
                        "start": w.get("start", 0),
                        "end": w.get("end", 0),
                    })
            duration = librosa.get_duration(path=str(fp))
            transcripts[str(fp)] = {
                "filename": fp.name,
                "text": result["text"].strip(),
                "words": words,
                "duration_sec": round(duration, 2),
            }
        except Exception as e:
            print(f"  Failed: {fp.name}: {e}")
            transcripts[str(fp)] = {
                "filename": fp.name, "text": "", "words": [], "duration_sec": 0,
            }

    with open(TRANSCRIPTS_JSON, "w", encoding="utf-8") as f:
        json.dump(transcripts, f, indent=2, ensure_ascii=False)
    print(f"Saved: {TRANSCRIPTS_JSON}")
    del model_w

elif W_TEXT > 0:
    print(f"Transcripts already exist: {TRANSCRIPTS_JSON}")
else:
    print("Text weight is 0 — skipping transcription.")

## 3. Text + Pause + Prosodic Features (if needed)
Skipped if features CSV already exists or if text weight is 0.

In [ ]:
need_text_features = (W_TEXT > 0) and not FEATURES_CSV.exists()

if need_text_features:
    sys.path.insert(0, str(Path(".").resolve()))
    from extract_features_company import (
        compute_text_features, compute_pause_features, compute_prosodic_features,
        _empty_text_features, _empty_pause_features, _empty_prosodic_features,
    )

    with open(TRANSCRIPTS_JSON, encoding="utf-8") as f:
        transcripts = json.load(f)

    # Build label map from GT if available
    label_map = {}
    if HAS_GT:
        fn_col = next((c for c in gt_df.columns if c.lower() in ("filename", "file", "name")), None)
        if fn_col and label_col:
            label_map = dict(zip(gt_df[fn_col], gt_df[label_col]))

    rows = []
    for filepath, t in tqdm(transcripts.items(), desc="Extracting text features"):
        text = t.get("text", "")
        words = t.get("words", [])
        filename = t.get("filename", Path(filepath).name)

        text_feats = compute_text_features(text)
        pause_feats = compute_pause_features(words) if words else _empty_pause_features()
        prosodic_feats = compute_prosodic_features(filepath) if os.path.exists(filepath) else _empty_prosodic_features()

        row = {
            "filepath": filepath,
            "filename": filename,
            "label_int": label_map.get(filename, -1),
            "duration_sec": t.get("duration_sec", 0),
            "text": text[:200],
        }
        row.update(text_feats)
        row.update(pause_feats)
        row.update(prosodic_feats)
        rows.append(row)

    feat_df = pd.DataFrame(rows)
    feat_df.to_csv(FEATURES_CSV, index=False)
    print(f"Saved: {FEATURES_CSV} ({len(feat_df)} rows)")

elif W_TEXT > 0:
    print(f"Text features already exist: {FEATURES_CSV}")
else:
    print("Text weight is 0 — skipping text features.")

## 4. WavLM Embeddings (if needed)
Skipped if WavLM CSV already exists or if WavLM weight is 0.

In [ ]:
need_wavlm = (W_WAVLM > 0) and not WAVLM_FEAT_CSV.exists()

if need_wavlm:
    import torch
    import librosa
    from transformers import AutoFeatureExtractor, WavLMModel

    print("Loading WavLM-base-plus...")
    fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
    wavlm_model = WavLMModel.from_pretrained("microsoft/wavlm-base-plus")
    wavlm_model = wavlm_model.eval().to("cpu")
    EMBED_DIM = wavlm_model.config.hidden_size

    SR = 16000
    MAX_DUR = 60

    embeddings = []
    filenames = []
    for fp in tqdm(audio_files, desc="Extracting WavLM embeddings"):
        try:
            audio, _ = librosa.load(str(fp), sr=SR, mono=True, duration=MAX_DUR)
            if len(audio) < SR:
                embeddings.append(np.zeros(EMBED_DIM))
            else:
                with torch.no_grad():
                    inputs = fe(audio, sampling_rate=SR, return_tensors="pt", padding=True)
                    outputs = wavlm_model(**inputs)
                    emb = outputs.last_hidden_state[0].mean(dim=0).numpy()
                embeddings.append(emb)
        except Exception as e:
            print(f"  Failed {fp.name}: {e}")
            embeddings.append(np.zeros(EMBED_DIM))
        filenames.append(fp.name)

    embed_cols = [f"wavlm_{i}" for i in range(EMBED_DIM)]
    wl_df = pd.DataFrame(embeddings, columns=embed_cols)
    wl_df["filename"] = filenames
    wl_df["filepath"] = [str(f) for f in audio_files]
    wl_df.to_csv(WAVLM_FEAT_CSV, index=False)
    print(f"Saved: {WAVLM_FEAT_CSV} ({len(wl_df)} rows x {EMBED_DIM} features)")

    del wavlm_model, fe
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

elif W_WAVLM > 0:
    print(f"WavLM features already exist: {WAVLM_FEAT_CSV}")
else:
    print("WavLM weight is 0 — skipping WavLM extraction.")

## 5. wav2vec2 Scores (if needed)
Runs ONNX inference on 5-sec windows. Skipped if weight is 0.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

w2v_scores = {}  # filename -> mean_p_read

if W_WAV2VEC2 > 0 and WAV2VEC2_ONNX and os.path.exists(WAV2VEC2_ONNX):
    import onnxruntime as ort
    import librosa

    print(f"Running wav2vec2 ONNX inference: {WAV2VEC2_ONNX}")
    sess = ort.InferenceSession(WAV2VEC2_ONNX, providers=["CPUExecutionProvider"])
    SR = 16000
    WINDOW = 5 * SR  # 80000 samples

    for fp in tqdm(audio_files, desc="wav2vec2 scoring"):
        try:
            audio, _ = librosa.load(str(fp), sr=SR, mono=True)
            if len(audio) < WINDOW:
                w2v_scores[fp.name] = 0.0
                continue
            p_reads = []
            for start in range(0, len(audio) - WINDOW + 1, WINDOW):
                chunk = audio[start:start+WINDOW].astype(np.float32).reshape(1, -1)
                logits = sess.run(None, {"input_values": chunk})[0]
                p_reads.append(sigmoid(logits.flatten()[0]))
            if p_reads:
                w2v_scores[fp.name] = float(np.mean(p_reads))
            else:
                w2v_scores[fp.name] = 0.0
        except Exception as e:
            print(f"  Failed {fp.name}: {e}")
            w2v_scores[fp.name] = 0.0

    print(f"Scored {len(w2v_scores)} files")
elif W_WAV2VEC2 > 0:
    print(f"WARNING: wav2vec2 ONNX not found at {WAV2VEC2_ONNX}. Scores will be 0.")
else:
    print("wav2vec2 weight is 0 — skipping.")

## 6. Ensemble Inference

In [ ]:
# Build master dataframe aligned by filename
filenames = [f.name for f in audio_files]
filepaths = [str(f) for f in audio_files]
master = pd.DataFrame({"filename": filenames, "filepath": filepaths})

# ── Text XGBoost scores ────────────────────────────────────
text_proba = np.zeros(len(master))
if W_TEXT > 0 and FEATURES_CSV.exists():
    TEXT_FEATURES = [
        "filler_rate", "filler_count", "repetition_rate", "repair_rate",
        "ttr", "mattr", "complex_word_rate", "avg_word_length",
        "n_words", "n_unique_words",
        "avg_sentence_length", "std_sentence_length", "fragment_rate", "n_sentences",
        "self_ref_rate", "discourse_marker_rate", "hedge_rate",
        "noun_rate", "verb_rate", "adj_rate",
    ]
    PAUSE_FEATURES = [
        "pause_mean", "pause_std", "pause_median", "pause_skew",
        "long_pause_rate", "pause_ratio", "n_pauses", "pause_regularity",
        "pause_before_content_ratio", "pause_before_function_ratio",
        "mid_phrase_pause_rate", "words_per_sec", "articulation_rate",
    ]
    PROSODIC_FEATURES = [
        "f0_mean", "f0_std", "f0_range", "f0_skew", "f0_slope",
        "energy_mean", "energy_std", "speaking_rate_std",
    ]
    ALL_TEXT = TEXT_FEATURES + PAUSE_FEATURES + PROSODIC_FEATURES

    feat_df = pd.read_csv(FEATURES_CSV)
    # Align by filename
    feat_df = feat_df.set_index("filename").reindex(filenames).reset_index()
    text_cols = [c for c in ALL_TEXT if c in feat_df.columns]

    if TEXT_XGBOOST_MODEL and os.path.exists(TEXT_XGBOOST_MODEL) and os.path.exists(TEXT_SCALER):
        t_model = xgb.XGBClassifier()
        t_model.load_model(TEXT_XGBOOST_MODEL)
        t_scaler = joblib.load(TEXT_SCALER)

        # Check feature count matches scaler
        if t_scaler.n_features_in_ == len(text_cols):
            X_text = feat_df[text_cols].fillna(0).values
            X_text_s = t_scaler.transform(X_text)
            text_proba = t_model.predict_proba(X_text_s)[:, 1]
            print(f"Text XGBoost: scored {len(text_proba)} files (mean={text_proba.mean():.4f})")
        else:
            print(f"WARNING: Scaler expects {t_scaler.n_features_in_} features, got {len(text_cols)}. Skipping text model.")
    else:
        print(f"WARNING: Text model/scaler not found. Text scores = 0.")

# ── WavLM XGBoost scores ───────────────────────────────────
wavlm_proba = np.zeros(len(master))
if W_WAVLM > 0 and WAVLM_FEAT_CSV.exists():
    wl_df = pd.read_csv(WAVLM_FEAT_CSV)
    wl_df = wl_df.set_index("filename").reindex(filenames).reset_index()
    wavlm_cols = [c for c in wl_df.columns if c.startswith("wavlm_")]

    if WAVLM_XGBOOST_MODEL and os.path.exists(WAVLM_XGBOOST_MODEL) and os.path.exists(WAVLM_SCALER):
        w_model = xgb.XGBClassifier()
        w_model.load_model(WAVLM_XGBOOST_MODEL)
        w_scaler = joblib.load(WAVLM_SCALER)

        X_wl = wl_df[wavlm_cols].fillna(0).values
        X_wl_s = w_scaler.transform(X_wl)
        wavlm_proba = w_model.predict_proba(X_wl_s)[:, 1]
        print(f"WavLM XGBoost: scored {len(wavlm_proba)} files (mean={wavlm_proba.mean():.4f})")
    else:
        print(f"WARNING: WavLM model/scaler not found. WavLM scores = 0.")

# ── wav2vec2 scores ────────────────────────────────────────
w2v_proba = np.array([w2v_scores.get(fn, 0.0) for fn in filenames])
if W_WAV2VEC2 > 0:
    print(f"wav2vec2: {len(w2v_proba)} files (mean={w2v_proba.mean():.4f})")

# ── Combined ───────────────────────────────────────────────
combined = W_TEXT * text_proba + W_WAVLM * wavlm_proba + W_WAV2VEC2 * w2v_proba

master["text_score"] = np.round(text_proba, 4)
master["wavlm_score"] = np.round(wavlm_proba, 4)
master["w2v_score"] = np.round(w2v_proba, 4)
master["combined_score"] = np.round(combined, 4)
master["pred_label"] = (combined >= THRESHOLD).astype(int)
master["pred_label_str"] = master["pred_label"].map({1: "cheating", 0: "not cheating"})

print(f"\nWeights: text={W_TEXT}, wavlm={W_WAVLM}, w2v={W_WAV2VEC2}")
print(f"Threshold: {THRESHOLD}")
print(f"Predictions: {(master['pred_label']==1).sum()} cheating, {(master['pred_label']==0).sum()} not cheating")

## 7. Attach GT Labels (if available)

In [ ]:
LABEL_MAP = {
    "read": 1, "Read": 1, "READ": 1, "cheating": 1, "Cheating": 1,
    "reading": 1, "Reading": 1, "yes": 1, "Yes": 1, "Y": 1, "1": 1, 1: 1,
    "spontaneous": 0, "Spontaneous": 0, "not cheating": 0, "Not Cheating": 0,
    "Not cheating": 0, "no": 0, "No": 0, "N": 0, "0": 0, 0: 0,
    "genuine": 0, "Genuine": 0,
}

if HAS_GT:
    fn_col = next((c for c in gt_df.columns if c.lower() in ("filename", "file", "name")), None)
    if fn_col:
        gt_map = {}
        for _, row in gt_df.iterrows():
            raw = row[label_col]
            mapped = LABEL_MAP.get(raw, LABEL_MAP.get(str(raw), -1))
            gt_map[row[fn_col]] = mapped

        master["gt_label"] = master["filename"].map(gt_map).fillna(-1).astype(int)
        labeled = master[master["gt_label"] >= 0]
        print(f"Matched GT labels: {len(labeled)}/{len(master)} files")
        print(f"  Cheating (GT):     {(labeled['gt_label']==1).sum()}")
        print(f"  Not cheating (GT): {(labeled['gt_label']==0).sum()}")
    else:
        print("Could not match GT — no filename column found.")
        HAS_GT = False
else:
    master["gt_label"] = -1
    print("No GT labels — predictions only.")

## 8. Metrics (if GT available)

In [ ]:
if HAS_GT:
    labeled = master[master["gt_label"] >= 0].copy()
    y_true = labeled["gt_label"].values
    y_pred = labeled["pred_label"].values
    y_score = labeled["combined_score"].values

    print(f"{'='*60}")
    print(f"METRICS (n={len(labeled)}, threshold={THRESHOLD})")
    print(f"Weights: text={W_TEXT}, wavlm={W_WAVLM}, w2v={W_WAV2VEC2}")
    print(f"{'='*60}")
    print(f"\nAccuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1:        {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print()
    print(classification_report(y_true, y_pred, target_names=["not cheating", "cheating"]))

    cm = confusion_matrix(y_true, y_pred)
    print(f"Confusion Matrix:")
    print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
    print(f"  FN={cm[1,0]}  TP={cm[1,1]}")

    # Misclassifications
    wrong = labeled[labeled["gt_label"] != labeled["pred_label"]]
    fp_df = wrong[wrong["pred_label"] == 1]
    fn_df = wrong[wrong["pred_label"] == 0]
    print(f"\nMisclassifications: {len(wrong)} ({len(fp_df)} FP, {len(fn_df)} FN)")
    if len(wrong) > 0:
        show = ["filename", "gt_label", "pred_label_str", "combined_score",
                "text_score", "wavlm_score", "w2v_score"]
        show = [c for c in show if c in wrong.columns]
        print(wrong[show].to_string(index=False))

    # Threshold sweep
    print(f"\n{'='*60}")
    print("THRESHOLD SWEEP")
    print(f"{'='*60}")
    print(f"{'thresh':>7s} {'prec':>7s} {'recall':>7s} {'f1':>7s} {'flagged':>8s} {'FP':>4s} {'FN':>4s}")
    print("-" * 50)
    best_t_f1, best_t = 0, 0
    for t in np.arange(0.10, 0.91, 0.05):
        preds = (y_score >= t).astype(int)
        f = f1_score(y_true, preds, zero_division=0)
        p = precision_score(y_true, preds, zero_division=0)
        r = recall_score(y_true, preds, zero_division=0)
        n_fp = int(((y_true == 0) & (preds == 1)).sum())
        n_fn = int(((y_true == 1) & (preds == 0)).sum())
        marker = " <-- best" if f > best_t_f1 else ""
        if f > best_t_f1:
            best_t_f1 = f
            best_t = t
        print(f"  {t:.2f}   {p:.4f}  {r:.4f}  {f:.4f}  {preds.sum():>6d}  {n_fp:>3d}  {n_fn:>3d}{marker}")
    print(f"\nBest threshold: {best_t:.2f} (F1={best_t_f1:.4f})")

else:
    print("No GT labels — skipping metrics.")

## 9. All Predictions

In [ ]:
# Display
show_cols = ["filename"]
if HAS_GT:
    show_cols.append("gt_label")
show_cols += ["pred_label_str", "combined_score", "text_score", "wavlm_score", "w2v_score"]
show_cols = [c for c in show_cols if c in master.columns]

print(f"\nAll predictions ({len(master)} files):")
print(master[show_cols].to_string(index=False))

# Save
out_path = audio_dir.parent / f"{folder_name}_predictions.csv"
master.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

## 10. Quick Compare: Different Weight Configs (if GT available)
Runs multiple weight configurations and shows which is best.

In [ ]:
if HAS_GT:
    labeled = master[master["gt_label"] >= 0]
    y_true = labeled["gt_label"].values
    t_scores = labeled["text_score"].values
    wl_scores = labeled["wavlm_score"].values
    w2_scores = labeled["w2v_score"].values

    configs = []
    # Grid search over weight triplets
    for wt in np.arange(0, 1.01, 0.1):
        for ww in np.arange(0, 1.01 - wt, 0.1):
            w2 = round(1.0 - wt - ww, 1)
            if w2 < 0:
                continue
            combined = round(wt, 1) * t_scores + round(ww, 1) * wl_scores + round(w2, 1) * w2_scores
            # Try multiple thresholds
            for thr in [0.35, 0.40, 0.45, 0.50, 0.55]:
                preds = (combined >= thr).astype(int)
                f = f1_score(y_true, preds, zero_division=0)
                p = precision_score(y_true, preds, zero_division=0)
                r = recall_score(y_true, preds, zero_division=0)
                configs.append({
                    "w_text": round(wt, 1), "w_wavlm": round(ww, 1),
                    "w_w2v": round(w2, 1), "threshold": thr,
                    "f1": round(f, 4), "prec": round(p, 4), "recall": round(r, 4),
                })

    configs_df = pd.DataFrame(configs).sort_values("f1", ascending=False)
    print("Top 15 weight+threshold configurations:")
    print(configs_df.head(15).to_string(index=False))
else:
    print("No GT labels — cannot compare configurations.")